In [ ]:
# pilot-to-full: node_2_2
# Original run: c386a3aa-a932-4ff1-9187-8d44e632a5db
# Hypothesis: Following a block switch in the dynamic routing task, the trial-by-trial shift in pre-stimulus basel...
# Scaled from N=5 pilot to N=63 sessions

import subprocess
import sys
import os

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

try:
    import pynwb
except ImportError:
    install('pynwb')

try:
    import sklearn
except ImportError:
    install('scikit-learn')

try:
    import scipy
except ImportError:
    install('scipy')

from pynwb import NWBHDF5IO

import fsspec as _fsspec

def _open_nwb(path):
    if str(path).startswith("s3://"):
        _fs = _fsspec.filesystem("s3", anon=True)
        return NWBHDF5IO(_fs.open(str(path), "rb"), "r", load_namespaces=True)
    return NWBHDF5IO(str(path), "r", load_namespaces=True)

import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

nwb_files = [
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/664851_2023-11-16.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/699847_2024-04-18.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703880_2024-04-17.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/708016_2024-04-30.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/667252_2023-09-26.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/737403_2024-09-24.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/733891_2024-09-17.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/676909_2023-12-12.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703333_2024-04-09.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/702136_2024-03-05.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/674562_2023-10-04.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703880_2024-04-15.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/690706_2023-11-29.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/690706_2023-11-27.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/667252_2023-09-28.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/715710_2024-07-18.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/714748_2024-06-25.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/681532_2023-10-16.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/676909_2023-12-11.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/744279_2025-01-13.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703882_2024-04-24.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/744279_2025-01-15.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/759434_2025-02-07.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/676909_2023-12-14.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/737403_2024-09-25.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/726088_2024-06-20.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/733891_2024-09-16.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/686176_2023-12-05.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/702136_2024-03-07.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/714748_2024-06-27.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/667252_2023-09-27.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/690706_2023-11-28.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/737403_2024-09-26.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/713655_2024-08-06.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/690706_2023-11-30.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703882_2024-04-25.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/686176_2023-12-04.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/713655_2024-08-07.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/714748_2024-06-26.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/733891_2024-09-20.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/686176_2023-12-07.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/726088_2024-06-21.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/686740_2023-10-26.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/715710_2024-07-17.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/708016_2024-05-01.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/676909_2023-12-13.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/699847_2024-04-17.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/733780_2024-08-26.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/741137_2024-10-11.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703882_2024-04-23.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/664851_2023-11-15.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/712815_2024-05-21.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/686740_2023-10-23.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/686740_2023-10-25.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/715710_2024-07-16.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/703333_2024-04-10.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/681532_2023-10-18.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/737403_2024-09-27.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/681532_2023-10-17.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/713655_2024-08-09.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/733891_2024-09-18.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/668755_2023-08-31.nwb",
    "s3://aind-scratch-data/dynamic-routing/cache/nwb/v0.0.273/733891_2024-09-19.nwb"
]

pfc_substrings = ['MOs', 'ACA', 'PL', 'ILA', 'ORB']
sensory_substrings = ['VISp', 'AUDp']

def is_in_regions(struct, regions):
    if not isinstance(struct, str):
        return False
    return any(r in struct for r in regions)

def get_spike_counts(spike_times, stim_times, window=(-0.5, 0.0)):
    counts = np.zeros((len(stim_times), len(spike_times)))
    start_times = stim_times + window[0]
    end_times = stim_times + window[1]
    for i, st in enumerate(spike_times):
        st = np.array(st)
        if st.ndim == 0 or len(st) == 0:
            continue
        left = np.searchsorted(st, start_times)
        right = np.searchsorted(st, end_times)
        counts[:, i] = right - left
    return counts

traj_pfc = []
traj_sens = []

MAX_POST_SWITCH_TRIALS = 15
STABLE_TRIALS_END = 15

for fpath in nwb_files:
    try:
        with _open_nwb(fpath) as io:
            nwb = io.read()
            # Compat: 2024+ sessions store trials in intervals/trials
            if nwb.trials is None and hasattr(nwb, 'intervals') and 'trials' in nwb.intervals:
                class _T:
                    def __init__(self, grp): self._grp = grp
                    def to_dataframe(self): return self._grp.to_dataframe()
                nwb.trials = _T(nwb.intervals['trials'])

            if nwb.units is None or 'structure' not in nwb.units.colnames and 'location' not in nwb.units.colnames:
                continue
            units_df = nwb.units.to_dataframe()
            # Compat: 2024+ sessions use 'location' instead of 'structure'
            if 'structure' not in units_df.columns and 'location' in units_df.columns:
                units_df['structure'] = units_df['location']

            pfc_mask = units_df['structure'].apply(lambda x: is_in_regions(x, pfc_substrings))
            sens_mask = units_df['structure'].apply(lambda x: is_in_regions(x, sensory_substrings))
            pfc_units = units_df[pfc_mask]
            sens_units = units_df[sens_mask]
            
            if nwb.trials is None:
                continue
            trials_df = nwb.trials.to_dataframe()
            if 'stim_start_time' not in trials_df.columns or 'rewarded_modality' not in trials_df.columns:
                continue
            
            valid_stim = ~trials_df['stim_start_time'].isna()
            trials_df = trials_df[valid_stim].reset_index(drop=True)
            stim_times = trials_df['stim_start_time'].values
            
            pfc_spike_times = [np.array(st) for st in pfc_units['spike_times'].values]
            sens_spike_times = [np.array(st) for st in sens_units['spike_times'].values]
            
            X_pfc = get_spike_counts(pfc_spike_times, stim_times, window=(-0.5, 0.0))
            X_sens = get_spike_counts(sens_spike_times, stim_times, window=(-0.5, 0.0))
            
            modality = trials_df['rewarded_modality'].astype(str).str.lower()
            is_vis = modality.apply(lambda x: 1 if 'vis' in x else (0 if 'aud' in x else -1)).values
            
            if 'block_index' in trials_df.columns:
                blocks = trials_df['block_index'].fillna(-1).values
            else:
                continue
                
            unique_blocks = np.unique(blocks)
            unique_blocks = unique_blocks[unique_blocks != -1]
            stable_trial_indices = []
            for b in unique_blocks:
                idx = np.where(blocks == b)[0]
                if len(idx) > STABLE_TRIALS_END:
                    stable_trial_indices.extend(idx[-STABLE_TRIALS_END:])
                else:
                    stable_trial_indices.extend(idx)
                    
            stable_trial_indices = np.array(stable_trial_indices)
            valid_stable = [i for i in stable_trial_indices if is_vis[i] != -1]
            y_stable = is_vis[valid_stable]
            
            if len(np.unique(y_stable)) < 2:
                continue
                
            X_pfc_stable = X_pfc[valid_stable]
            X_sens_stable = X_sens[valid_stable]
            
            svm_pfc = None
            if X_pfc.shape[1] > 0:
                scaler_pfc = StandardScaler()
                X_pfc_scaled = scaler_pfc.fit_transform(X_pfc_stable)
                svm_pfc = LinearSVC(max_iter=10000, dual=False, class_weight='balanced')
                svm_pfc.fit(X_pfc_scaled, y_stable)
                dv_pfc_stable = svm_pfc.decision_function(X_pfc_scaled)
                scale_pfc = (np.mean(dv_pfc_stable[y_stable == 1]) - np.mean(dv_pfc_stable[y_stable == 0])) / 2.0
                if scale_pfc == 0: scale_pfc = 1.0
                
            svm_sens = None
            if X_sens.shape[1] > 0:
                scaler_sens = StandardScaler()
                X_sens_scaled = scaler_sens.fit_transform(X_sens_stable)
                svm_sens = LinearSVC(max_iter=10000, dual=False, class_weight='balanced')
                svm_sens.fit(X_sens_scaled, y_stable)
                dv_sens_stable = svm_sens.decision_function(X_sens_scaled)
                scale_sens = (np.mean(dv_sens_stable[y_stable == 1]) - np.mean(dv_sens_stable[y_stable == 0])) / 2.0
                if scale_sens == 0: scale_sens = 1.0
                
            if 'is_block_switch' in trials_df.columns:
                switch_indices = np.where(trials_df['is_block_switch'].fillna(False).astype(bool).values)[0]
            elif 'block_index' in trials_df.columns:
                switch_indices = np.where(np.diff(blocks) != 0)[0] + 1
            else:
                continue
                
            switch_indices = switch_indices[switch_indices > 0]
                
            for sw_idx in switch_indices:
                if is_vis[sw_idx] == -1:
                    continue
                end_idx = min(sw_idx + MAX_POST_SWITCH_TRIALS, len(trials_df))
                next_switches = switch_indices[switch_indices > sw_idx]
                if len(next_switches) > 0:
                    end_idx = min(end_idx, next_switches[0])
                    
                idx_seq = np.arange(sw_idx, end_idx)
                new_context = is_vis[sw_idx]
                sign = 1 if new_context == 1 else -1
                
                traj_p = np.full(MAX_POST_SWITCH_TRIALS, np.nan)
                if svm_pfc is not None:
                    X_seq_pfc = scaler_pfc.transform(X_pfc[idx_seq])
                    dv_pfc = (svm_pfc.decision_function(X_seq_pfc) / scale_pfc) * sign
                    traj_p[:len(dv_pfc)] = dv_pfc
                    
                traj_s = np.full(MAX_POST_SWITCH_TRIALS, np.nan)
                if svm_sens is not None:
                    X_seq_sens = scaler_sens.transform(X_sens[idx_seq])
                    dv_sens = (svm_sens.decision_function(X_seq_sens) / scale_sens) * sign
                    traj_s[:len(dv_sens)] = dv_sens
                    
                if svm_pfc is not None or svm_sens is not None:
                    traj_pfc.append(traj_p)
                    traj_sens.append(traj_s)
    except Exception as e:
        print(f"Error processing {fpath}: {e}")

traj_pfc = np.array(traj_pfc)
traj_sens = np.array(traj_sens)

print("Number of paired block switches analyzed:", len(traj_pfc))

if len(traj_pfc) == 0:
    print("No valid block switches found.")
    sys.exit(0)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    mean_pfc = np.nanmean(traj_pfc, axis=0)
    mean_sens = np.nanmean(traj_sens, axis=0)

def exp_func(t, A, B, tau):
    return A - B * np.exp(-t / tau)

def fit_tau(mean_dv):
    t = np.arange(len(mean_dv))
    valid = ~np.isnan(mean_dv)
    t_val = t[valid]
    y_val = mean_dv[valid]
    
    if len(t_val) < 4:
        return np.nan, None
        
    A_guess = y_val[-1]
    B_guess = A_guess - y_val[0]
    tau_guess = 3.0
    
    try:
        popt, _ = curve_fit(exp_func, t_val, y_val, 
                            p0=[A_guess, B_guess, tau_guess], 
                            bounds=([-np.inf, -np.inf, 0.1], [np.inf, np.inf, 100.0]),
                            maxfev=10000)
        return popt[2], popt
    except:
        return np.nan, None

tau_pfc, popt_pfc = fit_tau(mean_pfc)
tau_sens, popt_sens = fit_tau(mean_sens)

print(f"Prefrontal adaptation time constant (tau): {tau_pfc:.2f} trials" if not np.isnan(tau_pfc) else "Prefrontal adaptation time constant (tau): Fit failed")
print(f"Sensory adaptation time constant (tau): {tau_sens:.2f} trials" if not np.isnan(tau_sens) else "Sensory adaptation time constant (tau): Fit failed")

# Permutation test
if not np.isnan(tau_pfc) and not np.isnan(tau_sens):
    obs_diff = tau_pfc - tau_sens
    n_permutations = 1000
    count = 0
    valid_perms = 0
    np.random.seed(42)
    for _ in range(n_permutations):
        swap = np.random.rand(len(traj_pfc)) > 0.5
        perm_pfc = np.where(swap[:, None], traj_sens, traj_pfc)
        perm_sens = np.where(swap[:, None], traj_pfc, traj_sens)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mean_perm_pfc = np.nanmean(perm_pfc, axis=0)
            mean_perm_sens = np.nanmean(perm_sens, axis=0)
            
        tp, _ = fit_tau(mean_perm_pfc)
        ts, _ = fit_tau(mean_perm_sens)
        
        if not np.isnan(tp) and not np.isnan(ts):
            valid_perms += 1
            if abs(tp - ts) >= abs(obs_diff):
                count += 1

    if valid_perms > 0:
        p_val = count / valid_perms
        print(f"Permutation test p-value: {p_val:.4f} (based on {valid_perms} valid permutations)")
    else:
        print("Permutation test failed due to curve fitting issues.")
else:
    print("Could not compute reliable time constants for permutation test.")

# Plotting
t = np.arange(MAX_POST_SWITCH_TRIALS)

plt.figure(figsize=(8, 6))
plt.plot(t, mean_pfc, 'o-', label='Prefrontal Cortex (Data)', color='blue')
if popt_pfc is not None:
    plt.plot(t, exp_func(t, *popt_pfc), '--', color='blue', alpha=0.7, label=f'PFC Fit (tau={tau_pfc:.2f})')

plt.plot(t, mean_sens, 's-', label='Primary Sensory (Data)', color='red')
if popt_sens is not None:
    plt.plot(t, exp_func(t, *popt_sens), '--', color='red', alpha=0.7, label=f'Sens Fit (tau={tau_sens:.2f})')

plt.xlabel('Trials since block switch')
plt.ylabel('Context Decision Variable (Aligned)')
plt.title('Trial-by-trial Context Adaptation in Pre-stimulus Baseline')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('autodiscovery-results/pilot-to-full/node_2_2/context_adaptation_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()